## import ##

In [2]:
from scipy.io import readsav
import matplotlib.colors as mcolors
import numpy as np
%matplotlib qt
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import RegularGridInterpolator
import pandas as pd
import sunpy.visualization.colormaps as spcm
import matplotlib.cm as cm
from scipy.ndimage import zoom
from matplotlib.colors import Normalize as norm 


In [3]:
# 讀取 IDL 的 .sav 檔案
data = readsav("C:/Users/chjan/hmi_output_CH1271_1p5.sav")

# # 列出所有變數名稱 
# print(data.keys())

# 取出 BP3DZ 資料
BP3DX = data["BP3DX"].T
BP3DY = data["BP3DY"].T
BP3DZ = data["BP3DZ"].T
print("shape:", BP3DX.shape, BP3DY.shape, BP3DZ.shape)

shape: (2244, 1651, 101) (2244, 1651, 101) (2244, 1651, 101)


In [4]:
hmi = pd.read_csv("../data_1271/hmi_CH1271_cropped_1p5_sp.csv")
hmi = hmi.to_numpy()
aia_df = pd.read_csv("../data_1271/aia_CH1271_cropped_1p5_sp.csv")
aia = aia_df.to_numpy()
# 調整AIA到跟HMI一樣(AIA較小)
zoom_factors = (hmi.shape[0] / aia.shape[0], hmi.shape[1] / aia.shape[1])
print(f"Original aia shape: {aia.shape}")
aia = zoom(aia, zoom_factors, order=1)
print(f"Resized aia shape: {aia.shape}")
aia_df = pd.DataFrame(aia)


Original aia shape: (1883, 1385)
Resized aia shape: (2244, 1651)


In [5]:
# 假設 Bx, By, Bz, X, Y, Z 為已知的磁場向量場
Bx, By, Bz = BP3DX, BP3DY, BP3DZ
# print(Bz[154][126])


In [19]:
# 定義空間格點 (假設 X, Y, Z 均勻分佈)
Nx, Ny, Nz = Bx.shape
x = np.linspace(0, Nx-1, Nx)  # X 軸範圍為 [0, Nx-1]
y = np.linspace(0, Ny-1, Ny)  # Y 軸範圍為 [0, Ny-1]
z = np.linspace(0, Nz-1, Nz)  # Z 軸範圍為 [0, Nz-1]
end_layer = 1000
d = 10 # every d layer save
divided_end = (Nz-1) * d/end_layer
noise_level = 15
def trace_fieldline(start_point, step=5, n_steps=2000):
    """
    使用 Euler 方法追蹤磁場線
    start_point: 初始點 (x, y, z)
    step: 每步移動距離
    n_steps: 最大步數
    """
    traj = [start_point]  # 存儲磁場線上的點
    point = np.array(start_point)
    
    for _ in range(n_steps):

        px, py, pz = np.round(point).astype(int)
        # 取得當前點的磁場值
        B_vector = np.array([
            Bx[px, py, pz],
            By[px, py, pz],
            Bz[px, py, pz]
        ]).flatten()
        # 若磁場過小，停止追蹤
        norm = np.linalg.norm(B_vector)
        if norm < 1e-6:
            print("braek")
            break

        # 歸一化方向，確保方向正確
        B_vector /= np.abs(norm)
        point = point + step * B_vector  # 沿磁場方向前進
        # 確保不超出範圍
        if not (x[0] <= point[0] <= x[-1] and y[0] <= point[1] <= y[-1] and z[0] <= point[2] <= z[-1]/divided_end):
            break
        traj.append(point.copy())
    return np.array(traj)

# 3D 繪圖
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# CH contour
X_grid_aia, Y_grid_aia = np.meshgrid(x, y, indexing='ij')
Z_grid_aia = np.zeros_like(X_grid_aia)
CH_threshold = 60
CH_mask = (aia <= CH_threshold+0.1) & (aia >= CH_threshold-0.1) 
# ax.scatter(X_grid_aia[CH_mask], Y_grid_aia[CH_mask], Z_grid_aia[CH_mask], c='b', label='AIA < 60', marker='o', s = 1)
ax.contour(X_grid_aia, Y_grid_aia, aia_df-CH_threshold, levels=[0], colors='black', linewidths=1)
CH_region_mask = aia <= CH_threshold
# 定義幾個磁場線的起點
# x, y 均勻分布在索引範圍內
x_range =  np.round(np.linspace(0, Nx-1, 77)[1:-1])  # 取 10 個點
y_range = np.round(np.linspace(0, Ny-1, 77)[1:-1])  # 取 10 個點

# 生成 (x, y) 網格，z=0
X_grid, Y_grid = np.meshgrid(x_range, y_range)
Z_grid = np.zeros_like(X_grid)  # 全部 z=0

# 組成 start_points
start_points = np.column_stack((X_grid.ravel(), Y_grid.ravel(), Z_grid.ravel()))
# print(CH_region_mask[int(start_points[0][0]),int(start_points[0][1])])
coord_list=[]
fs_list=[]
cmap = cm.get_cmap('spring')
norm = mcolors.Normalize(vmin=1, vmax=10)
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
color = cmap(norm(10))
for sp in start_points:
    traj = trace_fieldline(sp)
    # break
    if not CH_region_mask[int(sp[0]),int(sp[1])] or sp[0]>2000:
        pass
        continue
    else:
        px, py, pz = np.round(traj[-1]).astype(int)
        spx, spy, spz = np.round(sp).astype(int)
        # print(sp)
        if pz>10:
            B0= (Bx[spx, spy, spz]**2+By[spx, spy, spz]**2+Bz[spx, spy, spz]**2)**(1/2)
            if B0 < noise_level:
                pass
                continue
            coord_list.append([spx,spy])
            fs = B0 / Bz[px,py,pz]/6.25
            color = cmap(norm(fs))
            fs_list.append(fs)
            # if fs<1:
            #     print(B0, Bz[px,py,pz])
            # ax.text(px, py, d*pz+10, f"{fs:.1f}", color='b', fontsize=8)
    if len(traj)>10:
        ax.plot3D(traj[:, 0], traj[:, 1], traj[:, 2]*d, color=color, linewidth=2)
print(F"mean of fs = {np.mean(fs_list)} (in {len(fs_list)})")

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_xlim(0, x[-1])
ax.set_ylim(0, y[-1])
ax.set_zlim(0, z[-1]*d/divided_end)
cbar = fig.colorbar(sm, ax=ax, shrink=0.5, aspect=10)
cbar.set_label("Expansion Factor (fs)", fontsize=12)
# ax.grid(False)
ax.w_xaxis.pane.fill = False
ax.w_yaxis.pane.fill = False
ax.w_zaxis.pane.fill = False
ax.set_title('CH1271 streamline')
# ax.minorticks_on()
ax.view_init(elev=30, azim=-50)
plt.show()


mean of fs = 2.2127235965970526 (in 196)


In [21]:
fig2, ax2 = plt.subplots(figsize=(6, 6))
for coord, fs in zip(coord_list,fs_list):
    cmap = cm.get_cmap('rainbow')
    norm = mcolors.Normalize(vmin=1, vmax=5)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    color = cmap(norm(fs))
    ax2.scatter(coord[1], coord[0], color=color, s=50)
# ax2.set_aspect('equal')
plt.gca().set_aspect(1)
ax2.grid(True)
cbar = fig2.colorbar(sm, ax=ax2, shrink=0.5, aspect=10)
cbar.set_label("Expansion Factor (fs)", fontsize=12)
ax2.contour(Y_grid_aia, X_grid_aia, aia_df-CH_threshold, levels=[0], 
            colors='w', linewidths=1)
ax2.invert_yaxis()
ax.grid()
# ax2.imshow(hmi, cmap = 'gray', vmin=-50, vmax=50)
sdoaia193 = spcm.cmlist["sdoaia193"]
img1 = ax2.imshow(aia, cmap=sdoaia193, vmin=0, vmax=600)
plt.show()

In [8]:
np.sort(fs_list)[-20:]

array([ 2.37897173,  2.44574986,  2.55578924,  2.60581777,  2.73001981,
        2.79761626,  2.81947108,  2.93815989,  3.00131506,  3.06097386,
        3.47664845,  3.6028736 ,  4.43796322,  4.79836271,  4.91486712,
        5.20689123,  5.54580672,  5.65516626,  6.30547925, 29.1777193 ])